# Frozen encoder benchmark
ResNet-18 → Phikon-v2 → UNI2-h, with the same 9,000 development patches and 7,180 independent test patches. Frozen features + validation-selected logistic regression. No fine-tuning.

**Colab:** choose a T4 GPU, then run in order. Downloads total ~12.5 GB; allow ~25 GB free for archives, selected images, checkpoints and caches. UNI2-h requires approved Hugging Face access. Results are uncalibrated, patch-level research measurements.


## 1. Setup
Keep Colab’s paired PyTorch / torchvision installation. All reusable code lives in `benchmark/` so the notebook stays short. The setup refreshes an existing checkout. Use a fresh runtime when upgrading from the earlier Transformers 4 setup; restart if pip requests it.


In [ ]:
from pathlib import Path
import os
import subprocess

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
REPO = Path("/content/patho_dash")
if not REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/aralcimcim/patho_dash.git", str(REPO)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
os.chdir(REPO)
print("Source commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())
%pip install -q -r requirements-colab.txt


## 2. Configure
Drive keeps caches and completed runs between sessions. Images and archives remain in temporary Colab storage. For gated weights, add a Colab secret named `HF_TOKEN` after accepting the model’s terms; the token is never exported. Start with ResNet-18 + Phikon-v2, then add UNI2-h after access is approved.


In [ ]:
USE_DRIVE = True
MODELS = ["resnet18", "phikon-v2", "uni2-h"]
BATCH_SIZES = {"resnet18": 64, "phikon-v2": 16, "uni2-h": 4}
WORKERS = 2
PRECISION = "fp16"  # CPU uses fp32. T4 does not require BF16.
REVISIONS = {}      # Optional immutable HF commit SHAs; resolved SHAs are recorded.

from google.colab import drive, userdata
if USE_DRIVE:
    drive.mount("/content/drive")
    PERSIST = Path("/content/drive/MyDrive/patho_benchmark")
else:
    PERSIST = Path("/content/patho_benchmark")
WORK = Path("/content/patho_work")
WORK.mkdir(exist_ok=True)
PERSIST.mkdir(parents=True, exist_ok=True)
for folder in ("cache", "runs"):
    target = PERSIST / folder
    target.mkdir(exist_ok=True)
    link = WORK / folder
    if not link.exists():
        link.symlink_to(target, target_is_directory=True)
try:
    token = userdata.get("HF_TOKEN")
except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
    token = None
if token:
    os.environ["HF_TOKEN"] = token
    del token

import torch
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (slow)")


## 3. Prepare shared data
Reuse the recovered historical training sample. The 80/20 validation split is stratified at patch level: no reliable patient/slide mapping is supplied. C selection never uses CRC-VAL-HE-7K. Source archives are checksum-verified; exact file duplicates across splits stop the run.

Training uses the original **Macenko-normalized NCT-CRC-HE-100K** archive, not NONORM. Test uses the original CRC-VAL-HE-7K archive; its record does not explicitly specify stain normalization. No new stain normalization is applied. Both sources report 224×224 pixels at 0.5 MPP.


In [ ]:
from benchmark.data import prepare
manifest = prepare(WORK, seed=42, validation_fraction=0.2)
display(manifest.groupby(["split", "class_code"]).size().unstack().reindex(columns=["ADI", "BACK", "DEB", "LYM", "MUC", "MUS", "NORM", "STR", "TUM"]))


## 4. Extract, select C, and evaluate
Each encoder stays frozen in eval/inference mode. Select C from `[0.001, 0.01, 0.1, 1, 10, 100]` by validation macro F1, refit on all development patches, then predict the independent test set. Matching embedding caches are reused. Missing model access is reported explicitly; no substitute weights are used.

If GPU memory runs out, lower that model’s batch size (try 1) and rerun. Non-finite FP16 features require an explicit FP32 rerun. Completed runs are immutable and reused on reruns; the cells below also find earlier completed runs from Drive. Historical C=1 scores are kept separate.


In [ ]:
from benchmark.run import run
new_runs = run(WORK, models=MODELS, batch_sizes=BATCH_SIZES,
               workers=WORKERS, precision=PRECISION, revisions=REVISIONS)


## 5. Review and export
Macro F1 is primary. The export also contains per-class and tumour metrics, normalized confusion matrices, probability scores, review curves, manifest hashes, checkpoint revisions, hardware and timing. Brier score is the mean per-image **sum** of nine squared probability errors. Extraction timing includes data loading and preprocessing, excluding model loading.


In [ ]:
import pandas as pd
from benchmark.publish import validate_run, bundle

completed = sorted((WORK / "runs").glob("*/run_metadata.json"))
runs = [path.parent for path in completed]
rows = []
for path in runs:
    meta, result = validate_run(path)
    rows.append({"model": meta["protocol"]["model_name"], "macro_f1": result["macro_f1"],
                 "accuracy": result["accuracy"], "tumour_sensitivity_0.50": result["tumour"]["sensitivity"],
                 "tumour_specificity_0.50": result["tumour"]["specificity"], "run_id": meta["run_id"]})
if rows:
    display(pd.DataFrame(rows).sort_values("macro_f1", ascending=False).reset_index(drop=True))
    output = bundle(runs, PERSIST / "dashboard-results.zip")
    from google.colab import files
    files.download(str(output))
else:
    print("No completed experiments to export. Check model access and runtime errors above.")


## 6. Add results to the dashboard
Extract `dashboard-results.zip` locally, then run from your `patho_dash` checkout:

```bash
python -m benchmark publish /path/to/extracted/*/
npm test
npm run serve
```

The importer validates IDs, canonical class order, probabilities, checksums and saved metrics before updating the model registry. Preview at `http://localhost:4179`, then commit `docs/data/models.json` and `docs/data/runs/` and push. GitHub Pages serves the results without inference or a backend. Only small exports are bundled—no images, embeddings, weights, tokens or fitted classifiers.

Comparisons require the exact same benchmark manifest. Patch validation cannot establish patient-level independence or absence of foundation-model pretraining contamination. This benchmark measures this dataset and protocol, not universal model quality.
